In [1]:
import zs_perturbation
from zs_perturbation import BENCHMARK_FILE, CONTROL, DATASET_NAME, TO_GENE_SYMBOL

In [2]:
zs_perturbation.download_dataset()

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [9]:
import scanpy as sc
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ScientaLab/eva-rna", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ScientaLab/eva-rna", trust_remote_code=True)

Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

In [2]:
import pandas as pd

In [3]:
df_benchmark = pd.read_csv(BENCHMARK_FILE, index_col=0)

In [4]:
diseases = df_benchmark["disease_abbrev"].unique()

In [5]:
import anndata
from pathlib import Path

for disease in diseases[1:2]:
    print(f"\nProcessing {disease}...")

    adata = anndata.read_h5ad(Path(DATASET_NAME) / disease / "dataset.h5ad")
    # print(adata)
    print(adata.obs["disease"].unique().tolist())
    print(adata.obs["tissue"].unique().tolist())


Processing CD...
['Crohn Disease', 'Control']
['Ileum']


In [6]:
genes = [x for l in df_benchmark.target_genes.to_list() for x in l.split(";")]

In [7]:
genes[1]

'BAFF'

In [10]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")
adata = adata[:, adata.var.highly_variable].copy()

/var/folders/rl/nsddz37s55zbfg5h7b060rlc0000gn/T/ipykernel_18716/1412411339.py:1: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")


In [11]:
device = "cpu"

batch_gene_ids = [adata.var_names.tolist()]
batch_expression = adata.X[:1]

inputs = tokenizer(batch_gene_ids, batch_expression, padding=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

In [150]:
z_holder = {}


def hook_fn(module, input, output):
    z_holder["z"] = output
    output.retain_grad()  # critical (non-leaf)


handle = model.layers[-2].register_forward_hook(hook_fn)

In [151]:
output = model.encode(**inputs)

predicted_expression = model.decode(output.gene_embeddings)

In [152]:
predicted_expression.sum().backward()

In [155]:
z_holder["z"].grad

tensor([[[-0.0885,  0.0556, -0.0829,  ...,  0.0043, -0.0523,  0.0721],
         [ 0.0036,  0.0589, -0.0242,  ..., -0.0062, -0.0119, -0.0064],
         [-0.0084,  0.0425, -0.0280,  ...,  0.0057, -0.0146, -0.0022],
         ...,
         [ 0.0751, -0.0364,  0.0356,  ..., -0.0188,  0.0414, -0.0131],
         [-0.0161,  0.1210, -0.0912,  ...,  0.0438, -0.0031, -0.0278],
         [-0.0144, -0.0236,  0.0129,  ...,  0.0032, -0.0098,  0.0204]]])